## Skip-gram (Word2Vec) dengan Gensim — 3 Kalimat + Klasifikasi Naive Bayes

## 1. Import Library

In [1]:
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Menyiapkan 3 Kalimat Berbeda

Ketiga kalimat ini bertema Jaringan Saraf Tiruan (JST), sehingga beberapa kata (misalnya *neuron*, *jaringan*, *data*) muncul berulang di lebih dari satu kalimat — ini penting agar Skip-gram punya cukup konteks untuk mempelajari hubungan antar kata.

In [2]:
kalimat = [
    "Jaringan saraf tiruan meniru cara kerja otak manusia untuk mengenali pola data",
    "Setiap neuron pada jaringan saraf tiruan menerima input lalu menghasilkan output melalui fungsi aktivasi",
    "Proses pelatihan jaringan saraf tiruan menggunakan data dalam jumlah besar agar model semakin akurat"
]

for i, k in enumerate(kalimat, 1):
    print(f"Kalimat {i}: {k}")

Kalimat 1: Jaringan saraf tiruan meniru cara kerja otak manusia untuk mengenali pola data
Kalimat 2: Setiap neuron pada jaringan saraf tiruan menerima input lalu menghasilkan output melalui fungsi aktivasi
Kalimat 3: Proses pelatihan jaringan saraf tiruan menggunakan data dalam jumlah besar agar model semakin akurat


## 3. Preprocessing & Tokenisasi

In [3]:
def tokenisasi(teks: str):
    teks = teks.lower()
    tokens = re.findall(r'[a-zA-Z]+', teks)
    return tokens

corpus = [tokenisasi(k) for k in kalimat]

for i, tokens in enumerate(corpus, 1):
    print(f"Kalimat {i} -> {tokens}")

semua_kata = sorted(set(w for tokens in corpus for w in tokens))
print(f"\nJumlah kata unik (vocabulary) di seluruh korpus: {len(semua_kata)}")
print(semua_kata)

Kalimat 1 -> ['jaringan', 'saraf', 'tiruan', 'meniru', 'cara', 'kerja', 'otak', 'manusia', 'untuk', 'mengenali', 'pola', 'data']
Kalimat 2 -> ['setiap', 'neuron', 'pada', 'jaringan', 'saraf', 'tiruan', 'menerima', 'input', 'lalu', 'menghasilkan', 'output', 'melalui', 'fungsi', 'aktivasi']
Kalimat 3 -> ['proses', 'pelatihan', 'jaringan', 'saraf', 'tiruan', 'menggunakan', 'data', 'dalam', 'jumlah', 'besar', 'agar', 'model', 'semakin', 'akurat']

Jumlah kata unik (vocabulary) di seluruh korpus: 33
['agar', 'aktivasi', 'akurat', 'besar', 'cara', 'dalam', 'data', 'fungsi', 'input', 'jaringan', 'jumlah', 'kerja', 'lalu', 'manusia', 'melalui', 'menerima', 'mengenali', 'menggunakan', 'menghasilkan', 'meniru', 'model', 'neuron', 'otak', 'output', 'pada', 'pelatihan', 'pola', 'proses', 'saraf', 'semakin', 'setiap', 'tiruan', 'untuk']


## 4. Pasangan Kata (Skip-gram Word Pairs)

Skip-gram mengambil 1 kata sebagai **kata target/pusat (center word)**, lalu mencoba memprediksi **kata-kata di sekitarnya (context word)** dalam jangkauan tertentu (*window*). Untuk setiap kata target, dibentuk **pasangan (target, context)** — pasangan inilah yang menjadi data latih. Di sini pasangan dibuat manual untuk memperlihatkan prosesnya (Gensim melakukan ini otomatis di dalam `Word2Vec(sg=1, window=...)`).

In [4]:
WINDOW = 2  # jangkauan konteks kiri/kanan, harus sama dengan window model nanti

def buat_pasangan_skipgram(tokens, window=WINDOW):
    pasangan = []
    for i, target in enumerate(tokens):
        mulai = max(0, i - window)
        akhir = min(len(tokens), i + window + 1)
        for j in range(mulai, akhir):
            if j != i:
                pasangan.append((target, tokens[j]))
    return pasangan

daftar_pasangan = []
for idx, tokens in enumerate(corpus, 1):
    pasangan = buat_pasangan_skipgram(tokens)
    for target, context in pasangan:
        daftar_pasangan.append({'Kalimat_ke': idx, 'Kata_Target': target, 'Kata_Konteks': context})

df_pasangan = pd.DataFrame(daftar_pasangan)
print(f"Total pasangan (target, context) yang terbentuk: {len(df_pasangan)} (window={WINDOW})\n")

for idx in [1, 2, 3]:
    print(f"--- Pasangan kata - Kalimat {idx} ---")
    display(df_pasangan[df_pasangan['Kalimat_ke'] == idx].reset_index(drop=True))
    print()

Total pasangan (target, context) yang terbentuk: 142 (window=2)

--- Pasangan kata - Kalimat 1 ---


,Kalimat_ke,Kata_Target,Kata_Konteks
0,1,jaringan,saraf
1,1,jaringan,tiruan
2,1,saraf,jaringan
3,1,saraf,tiruan
4,1,saraf,meniru
5,1,tiruan,jaringan
6,1,tiruan,saraf
7,1,tiruan,meniru
8,1,tiruan,cara
9,1,meniru,saraf



--- Pasangan kata - Kalimat 2 ---


,Kalimat_ke,Kata_Target,Kata_Konteks
0,2,setiap,neuron
1,2,setiap,pada
2,2,neuron,setiap
3,2,neuron,pada
4,2,neuron,jaringan
5,2,pada,setiap
6,2,pada,neuron
7,2,pada,jaringan
8,2,pada,saraf
9,2,jaringan,neuron



--- Pasangan kata - Kalimat 3 ---


,Kalimat_ke,Kata_Target,Kata_Konteks
0,3,proses,pelatihan
1,3,proses,jaringan
2,3,pelatihan,proses
3,3,pelatihan,jaringan
4,3,pelatihan,saraf
5,3,jaringan,proses
6,3,jaringan,pelatihan
7,3,jaringan,saraf
8,3,jaringan,tiruan
9,3,saraf,pelatihan


In [5]:
df_pasangan.to_csv('pasangan_kata_skipgram.csv', index=False)
print("Tersimpan: 'pasangan_kata_skipgram.csv' ->", df_pasangan.shape)

Tersimpan: 'pasangan_kata_skipgram.csv' -> (142, 3)


## 5. Melatih Model Skip-gram dengan Gensim

- **`sg=1`** -> arsitektur **Skip-gram** (`sg=0` = CBOW).
- **`vector_size`** -> dimensi vektor tiap kata (50, karena korpus kecil).
- **`window`** -> disamakan dengan `WINDOW` di atas.
- **`min_count=1`** -> tetap sertakan kata walau cuma muncul 1 kali.

In [6]:
model = Word2Vec(
    sentences=corpus,
    vector_size=50,
    window=WINDOW,
    min_count=1,
    sg=1,          # sg=1 -> Skip-gram (bukan CBOW)
    epochs=200,
    seed=42,
    workers=1
)

print("Model Skip-gram berhasil dilatih.")
print("Jumlah kata dalam vocabulary model:", len(model.wv.index_to_key))
print("Dimensi vektor tiap kata:", model.vector_size)

Model Skip-gram berhasil dilatih.
Jumlah kata dalam vocabulary model: 33
Dimensi vektor tiap kata: 50


## 6. Representasi Vektor (Faktor) Setiap Kata per Kalimat

In [7]:
for i, tokens in enumerate(corpus, 1):
    print("="*80)
    print(f"KALIMAT {i}: {kalimat[i-1]}")
    print("="*80)
    for kata in tokens:
        vektor = model.wv[kata]
        print(f"\nKata: '{kata}'")
        print(f"Vektor (5 nilai pertama dari {len(vektor)} dimensi): {np.round(vektor[:5], 4)} ...")
    print()

KALIMAT 1: Jaringan saraf tiruan meniru cara kerja otak manusia untuk mengenali pola data

Kata: 'jaringan'
Vektor (5 nilai pertama dari 50 dimensi): [ 0.0197 -0.0205  0.0109 -0.007   0.0127] ...

Kata: 'saraf'
Vektor (5 nilai pertama dari 50 dimensi): [ 0.0159 -0.0204 -0.0067  0.0099  0.0011] ...

Kata: 'tiruan'
Vektor (5 nilai pertama dari 50 dimensi): [-0.0114  0.0036  0.0065  0.0088 -0.0026] ...

Kata: 'meniru'
Vektor (5 nilai pertama dari 50 dimensi): [-0.0058  0.0054  0.0126 -0.002  -0.015 ] ...

Kata: 'cara'
Vektor (5 nilai pertama dari 50 dimensi): [-0.0058  0.0087  0.0011  0.0121 -0.0013] ...

Kata: 'kerja'
Vektor (5 nilai pertama dari 50 dimensi): [ 0.0041 -0.0147 -0.0115 -0.0008  0.0045] ...

Kata: 'otak'
Vektor (5 nilai pertama dari 50 dimensi): [-0.0052 -0.0251 -0.019  -0.002   0.0034] ...

Kata: 'manusia'
Vektor (5 nilai pertama dari 50 dimensi): [ 0.0165 -0.0155 -0.009  -0.0007 -0.0127] ...

Kata: 'untuk'
Vektor (5 nilai pertama dari 50 dimensi): [-0.0148 -0.0108 -0.0008

### 6b. Ditampilkan sebagai Tabel (DataFrame) per Kalimat

In [8]:
kolom_dim = [f'dim_{i+1}' for i in range(8)]

for i, tokens in enumerate(corpus, 1):
    print(f"--- Tabel Vektor Kata - Kalimat {i} ---")
    data_tabel = []
    for kata in tokens:
        vektor = model.wv[kata]
        data_tabel.append([kata] + list(np.round(vektor[:8], 4)))
    df_kalimat = pd.DataFrame(data_tabel, columns=['Kata'] + kolom_dim)
    display(df_kalimat)
    print()

--- Tabel Vektor Kata - Kalimat 1 ---


,Kata,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8
0,jaringan,0.0197,-0.0205,0.0109,-0.0070,0.0127,0.0245,-0.0002,0.0130
1,saraf,0.0159,-0.0204,-0.0067,0.0099,0.0011,-0.0061,-0.0096,-0.0074
2,tiruan,-0.0114,0.0036,0.0065,0.0088,-0.0026,0.0266,-0.0272,0.0138
3,meniru,-0.0058,0.0054,0.0126,-0.0020,-0.0150,0.0207,-0.0186,0.0066
4,cara,-0.0058,0.0087,0.0011,0.0121,-0.0013,-0.0067,0.0097,0.0145
5,kerja,0.0041,-0.0147,-0.0115,-0.0008,0.0045,-0.0040,0.0102,0.0028
6,otak,-0.0052,-0.0251,-0.0190,-0.0020,0.0034,-0.0095,0.0116,0.0022
7,manusia,0.0165,-0.0155,-0.0090,-0.0007,-0.0127,-0.0081,-0.0123,0.0187
8,untuk,-0.0148,-0.0108,-0.0008,0.0249,-0.0115,0.0078,-0.0135,-0.0106
9,mengenali,0.0131,-0.0101,0.0071,0.0231,0.0049,0.0137,-0.0166,-0.0069



--- Tabel Vektor Kata - Kalimat 2 ---


,Kata,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8
0,setiap,0.0178,-0.0059,0.0193,0.0174,0.0059,0.0237,-0.0050,0.0121
1,neuron,-0.0024,-0.0057,-0.0030,0.0248,-0.0080,0.0123,-0.0225,0.0179
2,pada,0.0019,-0.0055,0.0166,-0.0073,-0.0082,-0.0053,-0.0161,0.0194
3,jaringan,0.0197,-0.0205,0.0109,-0.0070,0.0127,0.0245,-0.0002,0.0130
4,saraf,0.0159,-0.0204,-0.0067,0.0099,0.0011,-0.0061,-0.0096,-0.0074
5,tiruan,-0.0114,0.0036,0.0065,0.0088,-0.0026,0.0266,-0.0272,0.0138
6,menerima,0.0034,-0.0122,-0.0002,0.0275,-0.0173,0.0297,-0.0098,0.0155
7,input,0.0156,0.0036,0.0024,0.0242,-0.0154,0.0264,0.0065,0.0044
8,lalu,-0.0124,-0.0150,0.0047,0.0070,0.0110,0.0154,-0.0125,0.0098
9,menghasilkan,-0.0168,-0.0248,-0.0111,0.0241,-0.0022,0.0186,-0.0111,-0.0092



--- Tabel Vektor Kata - Kalimat 3 ---


,Kata,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8
0,proses,0.0016,0.0031,-0.0125,0.0164,0.0025,0.0056,0.0063,0.0078
1,pelatihan,0.0082,0.0065,0.0085,-0.0087,0.0143,0.0074,-0.0130,0.0039
2,jaringan,0.0197,-0.0205,0.0109,-0.0070,0.0127,0.0245,-0.0002,0.0130
3,saraf,0.0159,-0.0204,-0.0067,0.0099,0.0011,-0.0061,-0.0096,-0.0074
4,tiruan,-0.0114,0.0036,0.0065,0.0088,-0.0026,0.0266,-0.0272,0.0138
5,menggunakan,0.0169,-0.0109,0.0125,0.0014,0.0206,0.0097,0.0008,0.0198
6,data,0.0191,-0.0159,0.0162,0.0008,-0.0017,0.0167,-0.0238,0.0071
7,dalam,-0.0101,0.0001,0.0180,0.0243,-0.0074,0.0078,-0.0073,-0.0055
8,jumlah,0.0238,0.0106,-0.0007,-0.0093,0.0058,0.0123,-0.0233,0.0103
9,besar,0.0247,0.0055,-0.0078,0.0278,0.0014,0.0087,-0.0109,-0.0097


## 7. Menyimpan Seluruh Representasi Vektor ke CSV

In [9]:
baris_csv = []
for i, tokens in enumerate(corpus, 1):
    for kata in tokens:
        vektor = model.wv[kata]
        baris = {'Kalimat_ke': i, 'Kalimat': kalimat[i-1], 'Kata': kata}
        for d in range(model.vector_size):
            baris[f'dim_{d+1}'] = vektor[d]
        baris_csv.append(baris)

df_vektor = pd.DataFrame(baris_csv)
df_vektor.to_csv('vektor_kata_skipgram.csv', index=False)

print("Tersimpan: 'vektor_kata_skipgram.csv' ->", df_vektor.shape)
df_vektor.iloc[:, :6].head(10)

Tersimpan: 'vektor_kata_skipgram.csv' -> (40, 53)


,Kalimat_ke,Kalimat,Kata,dim_1,dim_2,dim_3
0,1,Jaringan saraf tiruan meniru cara kerja otak m...,jaringan,0.019716,-0.020456,0.010936
1,1,Jaringan saraf tiruan meniru cara kerja otak m...,saraf,0.015913,-0.020379,-0.006741
2,1,Jaringan saraf tiruan meniru cara kerja otak m...,tiruan,-0.011392,0.003566,0.006534
3,1,Jaringan saraf tiruan meniru cara kerja otak m...,meniru,-0.005809,0.005390,0.012589
4,1,Jaringan saraf tiruan meniru cara kerja otak m...,cara,-0.005793,0.008657,0.001092
5,1,Jaringan saraf tiruan meniru cara kerja otak m...,kerja,0.004148,-0.014674,-0.011522
6,1,Jaringan saraf tiruan meniru cara kerja otak m...,otak,-0.005203,-0.025088,-0.019046
7,1,Jaringan saraf tiruan meniru cara kerja otak m...,manusia,0.016541,-0.015492,-0.009046
8,1,Jaringan saraf tiruan meniru cara kerja otak m...,untuk,-0.014848,-0.010766,-0.000774
9,1,Jaringan saraf tiruan meniru cara kerja otak m...,mengenali,0.013115,-0.010143,0.007078


## 8. Uji Kualitas Embedding (Most Similar)

Karena korpusnya sangat kecil (cuma 3 kalimat), hasilnya belum tentu 100% masuk akal secara semantik — wajar untuk korpus sekecil ini.

In [10]:
kata_uji = ['jaringan', 'neuron', 'data']

for kata in kata_uji:
    print(f"Kata paling mirip dengan '{kata}':")
    mirip = model.wv.most_similar(kata, topn=3)
    for kata_mirip, skor in mirip:
        print(f"  - {kata_mirip}: {skor:.4f}")
    print()

Kata paling mirip dengan 'jaringan':
  - data: 0.6513
  - saraf: 0.6455
  - input: 0.6154

Kata paling mirip dengan 'neuron':
  - akurat: 0.4527
  - lalu: 0.4263
  - menghasilkan: 0.4100

Kata paling mirip dengan 'data':
  - jaringan: 0.6513
  - pelatihan: 0.6487
  - saraf: 0.6433



## 9. Klasifikasi Jenis Kata dengan Naive Bayes (Fitur: Vektor Skip-gram)

Vektor tiap kata (hasil Skip-gram) dipakai sebagai **fitur input** untuk classifier **Naive Bayes** (`GaussianNB`, cocok untuk fitur numerik kontinu seperti vektor skip-gram), untuk memprediksi **jenis kata (part-of-speech)**: `benda`, `kerja`, `sifat`, atau `lainnya`. Label diberikan manual berdasarkan tata bahasa untuk seluruh 33 kata unik.

In [11]:
label_jenis_kata = {
    'jaringan': 'benda', 'saraf': 'benda', 'tiruan': 'sifat', 'meniru': 'kerja',
    'cara': 'benda', 'kerja': 'benda', 'otak': 'benda', 'manusia': 'benda',
    'untuk': 'lainnya', 'mengenali': 'kerja', 'pola': 'benda', 'data': 'benda',
    'setiap': 'lainnya', 'neuron': 'benda', 'pada': 'lainnya', 'menerima': 'kerja',
    'input': 'benda', 'lalu': 'lainnya', 'menghasilkan': 'kerja', 'output': 'benda',
    'melalui': 'lainnya', 'fungsi': 'benda', 'aktivasi': 'benda', 'proses': 'benda',
    'pelatihan': 'benda', 'menggunakan': 'kerja', 'dalam': 'lainnya', 'jumlah': 'benda',
    'besar': 'sifat', 'agar': 'lainnya', 'model': 'benda', 'semakin': 'lainnya',
    'akurat': 'sifat',
}

assert set(label_jenis_kata.keys()) == set(semua_kata), "Ada kata yang belum diberi label!"

df_label = pd.DataFrame({
    'Kata': list(label_jenis_kata.keys()),
    'Jenis_Kata': list(label_jenis_kata.values())
})
print("Distribusi label jenis kata:")
print(df_label['Jenis_Kata'].value_counts())
df_label

Distribusi label jenis kata:


Jenis_Kata
benda      17
lainnya     8
kerja       5
sifat       3
Name: count, dtype: int64


,Kata,Jenis_Kata
0,jaringan,benda
1,saraf,benda
2,tiruan,sifat
3,meniru,kerja
4,cara,benda
5,kerja,benda
6,otak,benda
7,manusia,benda
8,untuk,lainnya
9,mengenali,kerja


In [12]:
X = np.array([model.wv[kata] for kata in df_label['Kata']])
y = df_label['Jenis_Kata'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Bentuk X (fitur):", X.shape)
print("Bentuk y (label):", y.shape)

Bentuk X (fitur): (33, 50)
Bentuk y (label): (33,)


**Catatan evaluasi:** karena vocabulary cuma 33 kata, dipakai **Leave-One-Out Cross-Validation (LOOCV)** supaya estimasi akurasi lebih bisa dipercaya dibanding split train/test biasa yang tidak stabil untuk data sekecil ini.

In [13]:
clf = GaussianNB()

loo = LeaveOneOut()
y_pred = cross_val_predict(clf, X_scaled, y, cv=loo)

akurasi = accuracy_score(y, y_pred)
print(f"Akurasi klasifikasi Naive Bayes (Leave-One-Out CV): {akurasi:.4f} ({akurasi*100:.1f}%)\n")

print("Classification Report:")
print(classification_report(y, y_pred, zero_division=0))

Akurasi klasifikasi Naive Bayes (Leave-One-Out CV): 0.3939 (39.4%)

Classification Report:
              precision    recall  f1-score   support

       benda       0.54      0.76      0.63        17
       kerja       0.00      0.00      0.00         5
     lainnya       0.00      0.00      0.00         8
       sifat       0.00      0.00      0.00         3

    accuracy                           0.39        33
   macro avg       0.14      0.19      0.16        33
weighted avg       0.28      0.39      0.33        33



In [14]:
df_hasil_klasifikasi = pd.DataFrame({
    'Kata': df_label['Kata'],
    'Jenis_Kata_Asli': y,
    'Jenis_Kata_Prediksi': y_pred,
    'Benar': y == y_pred
})
df_hasil_klasifikasi.to_csv('klasifikasi_jenis_kata_naive_bayes.csv', index=False)
print("Tersimpan: 'klasifikasi_jenis_kata_naive_bayes.csv' ->", df_hasil_klasifikasi.shape)
df_hasil_klasifikasi

Tersimpan: 'klasifikasi_jenis_kata_naive_bayes.csv' -> (33, 4)


,Kata,Jenis_Kata_Asli,Jenis_Kata_Prediksi,Benar
0,jaringan,benda,benda,True
1,saraf,benda,benda,True
2,tiruan,sifat,lainnya,False
3,meniru,kerja,lainnya,False
4,cara,benda,lainnya,False
5,kerja,benda,benda,True
6,otak,benda,benda,True
7,manusia,benda,benda,True
8,untuk,lainnya,benda,False
9,mengenali,kerja,lainnya,False


## 10. Kesimpulan

- Model **Skip-gram** (Gensim `Word2Vec`, `sg=1`) berhasil melatih vektor untuk 33 kata unik dari 3 kalimat.
- **Pasangan kata (target, context)** berhasil dibentuk manual (134 pasangan, window=2), tersimpan di `pasangan_kata_skipgram.csv`.
- Setiap kata berhasil dinyatakan sebagai **vektor (faktor) numerik** 50 dimensi, tersimpan di `vektor_kata_skipgram.csv`.
- Vektor tersebut dipakai sebagai fitur untuk **klasifikasi jenis kata dengan Naive Bayes (GaussianNB)**, dievaluasi dengan Leave-One-Out CV, hasil tersimpan di `klasifikasi_jenis_kata_naive_bayes.csv`.
- Data yang sangat kecil (3 kalimat) membatasi akurasi - lihat file kedua (`skipgram_klasifikasi_berita_detik.ipynb`) untuk versi dengan data berita ASLI dari Detik.com dalam jumlah lebih besar.